# 🏥 TopoNet Ablation Study & Replication (MICCAI 2025)
### Laparoscopic Liver Landmark Detection on L3D Dataset
**Automated End-to-End Kaggle GPU Notebook**

---

### Key Goals & Design
1. **Paper Replication (Table 2 & Table 1):**
   - Replicate the 6 ablation modes on the Validation split (122 frames).
   - Evaluate the full model on the Test split (109 frames) to benchmark against paper SOTA.
2. **Patient 32 4K Canvas Bug Fix:**
   - Automatically reads `imageHeight` and `imageWidth` directly from Label JSONs to avoid coordinate truncation on 4K images.
3. **Patient 40 Failure Analysis:**
   - Automatically isolates and logs metrics for Patient 40, and renders 4-panel visual diagnostic images (`RGB`, `Ground Truth`, `TopoNet Pred`, `Error Map`).
4. **Automated Result Packaging:**
   - Automatically packages all checkpoints, metrics CSVs, JSON summaries, and visual diagnostic panels into `/kaggle/working/EXPERIMENT_1_RESULTS.zip` for instant one-click download.


## Step 1: Environment & GPU Acceleration Check
Verify CUDA availability, GPU device specifications, and workspace folders.


In [ ]:
import os
import sys
import glob
import subprocess
import torch

print("=" * 70)
print("🚀 SYSTEM & GPU DIAGNOSTICS")
print("=" * 70)
print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Active GPU     : {gpu_name}")
    print(f"Total VRAM     : {total_mem:.2f} GB")
else:
    print("⚠️ WARNING: Running on CPU! Make sure GPU Accelerator is turned ON in Kaggle sidebar!")
print("=" * 70)

# Create structured workspace directories
os.makedirs('/kaggle/working/repos', exist_ok=True)
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/results', exist_ok=True)
os.makedirs('/kaggle/working/experiments/EXPERIMENT_1/models', exist_ok=True)
os.makedirs('/kaggle/working/experiments/EXPERIMENT_1/utils', exist_ok=True)
os.makedirs('/kaggle/working/experiments/EXPERIMENT_1/scripts', exist_ok=True)
print("✅ Workspace directories initialized under /kaggle/working/")


## Step 2: Install Dependencies & Compile Betti Matching 3D
Install surface distance evaluation packages and compile the C++ persistent homology library (`Betti-Matching-3D`).


In [ ]:
# 1. Install evaluation & vision libraries
!pip install -q surface-distance medpy einops timm

# 2. Build Betti Matching 3D C++ pybind module
print("📦 Compiling Betti Matching 3D C++ module...")
!mkdir -p /kaggle/working/betti_match/Betti_Matching
if not os.path.exists('/kaggle/working/betti_match/Betti_Matching/CMakeLists.txt'):
    !git clone --depth 1 https://github.com/nstucki/Betti-Matching-3D.git /kaggle/working/betti_match/Betti_Matching

%cd /kaggle/working/betti_match/Betti_Matching
!mkdir -p betti_build
%cd betti_build
!cmake .. -DCMAKE_BUILD_TYPE=Release > /dev/null
!make -j4 > /dev/null
%cd /kaggle/working

# Touch __init__.py files for clean Python module imports
!touch /kaggle/working/betti_match/__init__.py
!touch /kaggle/working/betti_match/Betti_Matching/__init__.py
!touch /kaggle/working/betti_match/Betti_Matching/betti_build/__init__.py

# Add to sys.path
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

# Verify Betti import
try:
    from betti_match.Betti_Matching.betti_build import betti_matching
    print("✅ Betti Matching C++ library compiled and imported successfully!")
except Exception as e:
    print(f"⚠️ Notice: Betti import check: {e}")


## Step 3: Clone Official TopoNet Codebase
Clone the official TopoNet repository into `/kaggle/working/repos/TopoNet` for reference modules (ResNet, DSCNet, PPM Decoder).


In [ ]:
if not os.path.exists('/kaggle/working/repos/TopoNet'):
    print("📥 Cloning cuiruize/TopoNet...")
    !git clone --depth 1 https://github.com/cuiruize/TopoNet.git /kaggle/working/repos/TopoNet
    print("✅ TopoNet reference repo cloned.")
else:
    print("✅ TopoNet repo already exists.")

if '/kaggle/working/repos/TopoNet' not in sys.path:
    sys.path.append('/kaggle/working/repos/TopoNet')


## Step 4: Download Depth Anything V2 ViT-B Weights
Download the official pretrained Depth Anything V2 ViT-B weights (`depth_anything_v2_vitb.pth`, ~390 MB).


In [ ]:
depth_ckpt = '/kaggle/working/checkpoints/depth_anything_v2_vitb.pth'
if not os.path.exists(depth_ckpt) or os.path.getsize(depth_ckpt) < 100_000_000:
    print("📥 Downloading Depth Anything V2 ViT-B weights from HuggingFace...")
    !wget -q --show-progress -O {depth_ckpt} "https://huggingface.co/depth-anything/Depth-Anything-V2-Base/resolve/main/depth_anything_v2_vitb.pth"
    print(f"✅ Depth weights saved to {depth_ckpt} ({os.path.getsize(depth_ckpt) / (1024**2):.1f} MB)")
else:
    print(f"✅ Depth weights already cached at {depth_ckpt}")


## Step 5: Automatic Dataset Path Discovery & Verification
Scan `/kaggle/input` to locate `Train`, `Val`, and `Test` directories and verify Patient 32 4K canvas handling.


In [ ]:
def discover_dataset_split(name):
    # Search common patterns on Kaggle
    patterns = [
        f'/kaggle/input/**/{name}',
        f'/kaggle/input/**/{name.lower()}',
        f'/kaggle/input/**/{name.upper()}',
        f'/kaggle/input/*{name.lower()}*/**/{name}',
        f'/kaggle/input/datasets/*/{name}',
    ]
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        for m in matches:
            if os.path.isdir(m) and (os.path.exists(os.path.join(m, 'images')) or os.path.exists(os.path.join(m, 'labels'))):
                return m
            elif os.path.isdir(m):
                # Check if images are inside
                imgs = glob.glob(os.path.join(m, '*.[jJ][pP][gG]'))
                if len(imgs) > 0:
                    return m
    return None

train_dir = discover_dataset_split('Train')
val_dir = discover_dataset_split('Val')
test_dir = discover_dataset_split('Test')

# Fallbacks if nested under specific user slug
if not train_dir:
    train_dir = '/kaggle/input/datasets/khoatrytopublish/l3d-train/Train'
if not val_dir:
    val_dir = '/kaggle/input/datasets/khoatrytopublish/l3d-val/Val'
if not test_dir:
    test_dir = '/kaggle/input/datasets/khoatrytopublish/l3d-test/Test'

print("=" * 70)
print("📂 DATASET SPLIT LOCATIONS")
print("=" * 70)
print(f"Train Dir: {train_dir} (Exists: {os.path.exists(train_dir)})")
print(f"Val Dir  : {val_dir} (Exists: {os.path.exists(val_dir)})")
print(f"Test Dir : {test_dir} (Exists: {os.path.exists(test_dir)})")
print("=" * 70)


## Step 6: Deploy Experiment Modules
Deploy decoupled Dataset with Patient 32 4K fix, Evaluation Metrics with ASSD, Unified TopoNet Ablation Model, and Training Runner.


In [ ]:
# 1. Write __init__.py files
!touch /kaggle/working/experiments/__init__.py
!touch /kaggle/working/experiments/EXPERIMENT_1/__init__.py
!touch /kaggle/working/experiments/EXPERIMENT_1/models/__init__.py
!touch /kaggle/working/experiments/EXPERIMENT_1/utils/__init__.py
!touch /kaggle/working/experiments/EXPERIMENT_1/scripts/__init__.py

# 2. Write dataset.py
with open('/kaggle/working/experiments/EXPERIMENT_1/utils/dataset.py', 'w') as f:
    f.write('''import os
import glob
import json
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from torchvision import transforms as T


class TopoNetDataset(Dataset):
    """
    Robust L3D Dataset Reader for TopoNet with dynamic canvas sizing.
    Fixes the Patient 32 4K canvas truncation bug by reading imageHeight/imageWidth
    directly from the Label JSON.
    """
    def __init__(self, data_dir, transform=None, mode='train'):
        self.data_dir = data_dir
        self.mode = mode
        
        # Resolve images directory flexibly (supports both nested 'images/' and flat folders)
        if os.path.exists(os.path.join(data_dir, 'images')):
            self.image_paths = sorted(glob.glob(os.path.join(data_dir, 'images', '*.[jJ][pP][gG]')) + 
                                     glob.glob(os.path.join(data_dir, 'images', '*.[pP][nN][gG]')))
        else:
            self.image_paths = sorted(glob.glob(os.path.join(data_dir, '*.[jJ][pP][gG]')) + 
                                     glob.glob(os.path.join(data_dir, '*.[pP][nN][gG]')))
            
        if len(self.image_paths) == 0:
            raise FileNotFoundError(f"No images found in dataset directory: {data_dir}")

        self.transform = transform if transform else self._default_transform

    @staticmethod
    def _default_transform(image, mask, depth):
        to_tensor = T.ToTensor()
        return to_tensor(image), to_tensor(mask), to_tensor(depth)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = self.load_image(img_path)
        mask = self.load_mask(img_path)
        depth = np.zeros((1024, 1024), dtype=np.uint8)  # TopoNet infers depth dynamically via depth_encoder

        # Transform expects mask in shape (H, W, C)
        image_t, mask_t, depth_t = self.transform(image, mask.transpose(1, 2, 0), depth)

        return image_t, depth_t, mask_t, os.path.basename(img_path)

    @staticmethod
    def load_image(path):
        img = cv2.imread(str(path))
        if img is None:
            raise FileNotFoundError(f"Could not load image at {path}")
        img = cv2.resize(img, (1024, 1024))
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    @staticmethod
    def load_mask(img_path):
        """
        Dynamically renders ground truth mask from the corresponding JSON label file.
        Uses exact image dimensions from JSON metadata to prevent coordinate clipping.
        """
        # Resolve label path
        if 'images' in str(img_path):
            json_path = str(img_path).replace('images', 'labels')
        else:
            parent = os.path.dirname(img_path)
            json_path = os.path.join(os.path.dirname(parent), 'labels', os.path.basename(img_path))
            
        json_path = os.path.splitext(json_path)[0] + '.json'

        if not os.path.exists(json_path):
            # Fallback: check alongside image
            json_path = os.path.splitext(str(img_path))[0] + '.json'
            if not os.path.exists(json_path):
                raise FileNotFoundError(f"Label JSON not found for image: {img_path}")

        # Load JSON and extract true image canvas dimensions
        with open(json_path, 'r') as f:
            data = json.load(f)

        # Dynamic canvas size: guarantees Patient 32 4K (2160x3840) is never truncated
        height = data.get('imageHeight', 1080)
        width = data.get('imageWidth', 1920)
        canvas = np.zeros((height, width), dtype=np.uint8)

        # Draw contours with thickness 35 (exact paper standard)
        for shape in data.get('shapes', []):
            points = shape.get('points', [])
            label = shape.get('label', '').lower().strip()

            # Class mapping: 1 = Ridge, 2 = Silhouette, 3 = Falciform Ligament
            if label.startswith('r'):
                color = 1
            elif label.startswith('s'):
                color = 2
            elif label.startswith('l'):
                color = 3
            else:
                color = 0

            for i in range(1, len(points)):
                pt1 = tuple(map(int, points[i - 1]))
                pt2 = tuple(map(int, points[i]))
                cv2.line(canvas, pt1, pt2, color, 35)

        # Resize to network input resolution (1024, 1024) using INTER_NEAREST
        canvas = cv2.resize(canvas, (1024, 1024), interpolation=cv2.INTER_NEAREST)

        # One-hot map into 4 channels: (0: BG, 1: Ridge, 2: Silhouette, 3: Falciform)
        masks = np.zeros((4, 1024, 1024), dtype=np.uint8)
        masks[0][canvas == 0] = 255
        masks[1][canvas == 1] = 255
        masks[2][canvas == 2] = 255
        masks[3][canvas == 3] = 255

        return masks
''')

# 3. Write metrics.py
with open('/kaggle/working/experiments/EXPERIMENT_1/utils/metrics.py', 'w') as f:
    f.write('''import numpy as np
import cv2
import torch


def compute_dice_iou(pred_binary, gt_binary):
    """
    Computes Dice Similarity Coefficient and IoU for binary 1D or 2D arrays.
    """
    intersection = np.logical_and(pred_binary, gt_binary).sum()
    pred_sum = pred_binary.sum()
    gt_sum = gt_binary.sum()
    total_sum = pred_sum + gt_sum

    if total_sum == 0:
        return 1.0, 1.0  # Perfect match on empty ground truth
    if pred_sum == 0 or gt_sum == 0:
        return 0.0, 0.0

    dice = (2.0 * intersection) / (total_sum + 1e-7)
    iou = intersection / (pred_sum + gt_sum - intersection + 1e-7)
    return float(dice), float(iou)


def compute_assd(pred_mask, gt_mask, fallback=80.0):
    """
    Computes Average Symmetric Surface Distance (ASSD) in pixels.
    Uses surface_distance / medpy if available, or robust OpenCV Euclidean distance transform fallback.
    """
    pred_mask = (pred_mask > 0).astype(np.uint8)
    gt_mask = (gt_mask > 0).astype(np.uint8)

    if pred_mask.sum() == 0 or gt_mask.sum() == 0:
        return float(fallback)

    # Try surface_distance / medpy first
    try:
        from surface_distance import metrics
        sd = metrics.compute_surface_distances(gt_mask.astype(bool), pred_mask.astype(bool), (1.0, 1.0))
        assd_val = metrics.compute_average_surface_distance(sd)[1]
        if np.isnan(assd_val) or assd_val > 500:
            return float(fallback)
        return float(assd_val)
    except Exception:
        pass

    try:
        import medpy.metric
        return float(medpy.metric.assd(pred_mask, gt_mask))
    except Exception:
        pass

    # Pure OpenCV Euclidean Distance Transform Fallback
    # Extract 1-pixel boundary contours
    contours_pred, _ = cv2.findContours(pred_mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    contours_gt, _ = cv2.findContours(gt_mask, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)

    border_pred = np.zeros_like(pred_mask)
    border_gt = np.zeros_like(gt_mask)

    cv2.drawContours(border_pred, contours_pred, -1, 1, 1)
    cv2.drawContours(border_gt, contours_gt, -1, 1, 1)

    # Distance to GT surface
    dist_to_gt = cv2.distanceTransform(1 - border_gt, cv2.DIST_L2, 5)
    # Distance to Pred surface
    dist_to_pred = cv2.distanceTransform(1 - border_pred, cv2.DIST_L2, 5)

    dist_pred_to_gt = dist_to_gt[border_pred == 1]
    dist_gt_to_pred = dist_to_pred[border_gt == 1]

    if len(dist_pred_to_gt) == 0 or len(dist_gt_to_pred) == 0:
        return float(fallback)

    assd_val = (dist_pred_to_gt.mean() + dist_gt_to_pred.mean()) / 2.0
    return float(min(assd_val, fallback))


def evaluate_batch(pred_logits, gt_masks):
    """
    Evaluates a batch of multi-class predictions against ground truth.
    pred_logits: Tensor of shape (B, 4, H, W)
    gt_masks: Tensor of shape (B, 4, H, W) where masks are one-hot (0: BG, 1: Ridge, 2: Sil, 3: Falc)
    
    Returns list of metric dicts per sample.
    """
    pred_classes = torch.argmax(pred_logits, dim=1).detach().cpu().numpy()  # (B, H, W)
    gt_classes = torch.argmax(gt_masks, dim=1).detach().cpu().numpy()        # (B, H, W)

    batch_metrics = []

    for b in range(pred_classes.shape[0]):
        p_map = pred_classes[b]
        g_map = gt_classes[b]

        # Per-class foreground metrics (1: Ridge, 2: Silhouette, 3: Falciform)
        class_dices = []
        class_ious = []
        class_assds = []

        for c, name in enumerate(['ridge', 'silhouette', 'falciform'], start=1):
            p_c = (p_map == c)
            g_c = (g_map == c)

            d, iou = compute_dice_iou(p_c, g_c)
            class_dices.append(d)
            class_ious.append(iou)

            if g_c.sum() > 0:
                assd_c = compute_assd(p_c, g_c)
                class_assds.append(assd_c)

        macro_dice = float(np.mean(class_dices))
        macro_iou = float(np.mean(class_ious))
        macro_assd = float(np.mean(class_assds)) if len(class_assds) > 0 else 80.0

        # Overall flattened foreground metric (exact repos/TopoNet/test.py standard)
        p_fg = (p_map > 0)
        g_fg = (g_map > 0)
        fg_dice, fg_iou = compute_dice_iou(p_fg, g_fg)
        fg_assd = compute_assd(p_fg, g_fg)

        batch_metrics.append({
            'macro_dice': macro_dice,
            'macro_iou': macro_iou,
            'macro_assd': macro_assd,
            'fg_dice': fg_dice,
            'fg_iou': fg_iou,
            'fg_assd': fg_assd,
            'ridge_dice': class_dices[0],
            'sil_dice': class_dices[1],
            'falc_dice': class_dices[2],
            'ridge_iou': class_ious[0],
            'sil_iou': class_ious[1],
            'falc_iou': class_ious[2],
        })

    return batch_metrics
''')

# 4. Write toponet_ablation.py
with open('/kaggle/working/experiments/EXPERIMENT_1/models/toponet_ablation.py', 'w') as f:
    f.write('''import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure repos/TopoNet is in sys.path
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), '../../../repos/TopoNet'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from depth_anything_v2.dpt import DepthAnythingV2
from models.resnet import ResNet34
from models.context_modules import get_context_module
from models.model_utils import ConvBNAct, Swish
from models.decoder import Decoder
from models.befusion import BeFusion
from DSCNet.ds_encoder import DSCNet_Encoder


def _safe_interpolate_area(x, size):
    """Area interpolation with automatic MPS CPU fallback for non-divisible sizes."""
    if x.device.type == 'mps':
        return F.interpolate(x.cpu(), size=size, mode='area').to(x.device)
    return F.interpolate(x, size=size, mode='area')


class SimpleConcatFusion(nn.Module):
    """
    Simple concatenation baseline replacing BTF (Boundary-Aware Topological Fusion).
    Merges RGB and depth feature maps via 1x1 convolution.
    """
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, rgb_feat, depth_feat, prev_feat=None):
        if prev_feat is not None and prev_feat.shape[2:] != rgb_feat.shape[2:]:
            prev_feat = F.interpolate(prev_feat, size=rgb_feat.shape[2:], mode='bilinear', align_corners=False)
        fused = self.conv(torch.cat([rgb_feat, depth_feat], dim=1))
        if prev_feat is not None and prev_feat.shape[1] == fused.shape[1]:
            fused = fused + prev_feat
        return fused, fused


class TopoNetAblationModel(nn.Module):
    """
    Unified TopoNet Model supporting all 6 official paper ablation modes:
      1. 'full': Full TopoNet (Snake DSCNet + BTF + clDice + Betti)
      2. 'baseline': Standard Conv + Simple Concat (No BTF, no topo losses)
      3. 'wo_lper': Snake DSCNet + BTF + Soft Dice + clDice (no Betti)
      4. 'wo_lcl': Snake DSCNet + BTF + Soft Dice + Betti (no clDice)
      5. 'wo_lper_lcl': Snake DSCNet + BTF + Soft Dice only (no topo loss)
      6. 'wo_btf': Snake DSCNet + Simple Concat + Soft Dice + clDice + Betti
    """
    def __init__(self, ablation_mode='full', depth_path=None, num_classes=4, height=1024, width=1024):
        super().__init__()
        self.ablation_mode = ablation_mode
        self.depth_path = depth_path

        # 1. Depth Anything V2 Foundation Model
        depth_configs = {
            'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]}
        }
        self.depth_encoder = DepthAnythingV2(**depth_configs['vitb'])
        if depth_path and os.path.exists(depth_path):
            state_dict = torch.load(depth_path, map_location='cpu')
            self.depth_encoder.load_state_dict(state_dict)
            print(f"✅ Loaded Depth Anything V2 weights from: {depth_path}")
        else:
            print(f"⚠️  Depth weights not found at '{depth_path}'. Initialized with random weights (OK for smoke tests).")
        self.depth_encoder.eval()
        self.depth_encoder.requires_grad_(False)

        # 2. Depth Feature Extractor (Snake DSCNet vs Standard Conv)
        self.use_snake = (ablation_mode != 'baseline')
        if self.use_snake:
            self.dsc_encoder = DSCNet_Encoder()
        else:
            # Baseline uses standard ResNet blocks for depth
            self.dsc_encoder = ResNet34(input_channels=3, pretrained_on_imagenet=False)

        # 3. RGB Encoder (ResNet-34)
        self.rgb_encoder = ResNet34(input_channels=3, pretrained_on_imagenet=False)

        # 4. Multi-Modal Fusion (BTF vs Simple Concat)
        self.use_btf = (ablation_mode not in ['baseline', 'wo_btf'])
        if self.use_btf:
            self.be0 = BeFusion(64, 512, 512, isFirst=True)
            self.be1 = BeFusion(self.rgb_encoder.down_4_channels_out, 256, 256)
            self.be2 = BeFusion(self.rgb_encoder.down_8_channels_out, 128, 128)
            self.be3 = BeFusion(self.rgb_encoder.down_16_channels_out, 64, 64)
            self.be4 = BeFusion(self.rgb_encoder.down_32_channels_out, 32, 32, isLast=True)
        else:
            self.be0 = SimpleConcatFusion(64)
            self.be1 = SimpleConcatFusion(self.rgb_encoder.down_4_channels_out)
            self.be2 = SimpleConcatFusion(self.rgb_encoder.down_8_channels_out)
            self.be3 = SimpleConcatFusion(self.rgb_encoder.down_16_channels_out)
            self.be4 = SimpleConcatFusion(self.rgb_encoder.down_32_channels_out)

        # Skip connections
        channels_decoder = [128, 128, 128]
        self.skip_layer1 = nn.Sequential(
            ConvBNAct(self.rgb_encoder.down_4_channels_out, channels_decoder[2], kernel_size=1, activation=nn.ReLU(inplace=True))
        )
        self.skip_layer2 = nn.Sequential(
            ConvBNAct(self.rgb_encoder.down_8_channels_out, channels_decoder[1], kernel_size=1, activation=nn.ReLU(inplace=True))
        )
        self.skip_layer3 = nn.Sequential(
            ConvBNAct(self.rgb_encoder.down_16_channels_out, channels_decoder[0], kernel_size=1, activation=nn.ReLU(inplace=True))
        )

        # Context Module & Decoder
        self.context_module, channels_after_context = get_context_module(
            'ppm', self.rgb_encoder.down_32_channels_out, channels_decoder[0],
            input_size=(height // 32, width // 32), activation=nn.ReLU(inplace=True), upsampling_mode='bilinear'
        )

        self.decoder = Decoder(
            channels_in=channels_after_context, channels_decoder=channels_decoder,
            activation=nn.ReLU(inplace=True), nr_decoder_blocks=[1, 1, 1],
            encoder_decoder_fusion='add', upsampling_mode='bilinear', num_classes=num_classes
        )

    def forward(self, image):
        # 1. On-the-fly depth estimation at (1022, 1022) [multiple of ViT patch size 14]
        with torch.no_grad():
            img_depth_in = _safe_interpolate_area(image, size=(1022, 1022))
            raw_depth = self.depth_encoder.infer_image(img_depth_in)
            depth_3ch = raw_depth.expand(-1, 3, -1, -1)

        # 2. Depth Feature Extraction
        if self.use_snake:
            d0, d1, d2, d3, depth_out = self.dsc_encoder(depth_3ch)
        else:
            # Baseline standard CNN depth encoder
            out_d = self.dsc_encoder.forward_first_conv(depth_3ch)
            d0 = out_d
            out_d = F.max_pool2d(out_d, kernel_size=3, stride=2, padding=1)
            d1 = self.dsc_encoder.forward_layer1(out_d)
            d2 = self.dsc_encoder.forward_layer2(d1)
            d3 = self.dsc_encoder.forward_layer3(d2)
            depth_out = self.dsc_encoder.forward_layer4(d3)

        # 3. RGB Feature Extraction & Progressive Multi-Modal Fusion
        out_rgb = self.rgb_encoder.forward_first_conv(image)
        skipf0, out_f0 = self.be0(out_rgb, _safe_interpolate_area(d0, size=out_rgb.shape[2:]))
        out_rgb = F.max_pool2d(out_rgb, kernel_size=3, stride=2, padding=1)

        # Block 1
        out_rgb = self.rgb_encoder.forward_layer1(out_rgb)
        skipf1, out_f1 = self.be1(out_rgb, _safe_interpolate_area(d1, size=out_rgb.shape[2:]), out_f0)
        skip1 = self.skip_layer1(skipf1)

        # Block 2
        out_rgb = self.rgb_encoder.forward_layer2(out_rgb)
        skipf2, out_f2 = self.be2(out_rgb, _safe_interpolate_area(d2, size=out_rgb.shape[2:]), out_f1)
        skip2 = self.skip_layer2(skipf2)

        # Block 3
        out_rgb = self.rgb_encoder.forward_layer3(out_rgb)
        skipf3, out_f3 = self.be3(out_rgb, _safe_interpolate_area(d3, size=out_rgb.shape[2:]), out_f2)
        skip3 = self.skip_layer3(skipf3)

        # Block 4
        out_rgb = self.rgb_encoder.forward_layer4(out_rgb)
        skipf4, out_f4 = self.be4(out_rgb, _safe_interpolate_area(depth_out, size=out_rgb.shape[2:]), out_f3)

        # Context Module & Decoder
        context_out = self.context_module(out_f4)
        decoder_outs, _ = self.decoder(enc_outs=[context_out, skip3, skip2, skip1])
        logits = F.log_softmax(decoder_outs, dim=1)

        return logits, raw_depth
''')

# 5. Write train_toponet.py
with open('/kaggle/working/experiments/EXPERIMENT_1/scripts/train_toponet.py', 'w') as f:
    f.write('''import os
import sys
import time
import json
import argparse
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader

# Add experiment and repo roots to PYTHONPATH
EXPERIMENT_DIR = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
WORKSPACE_ROOT = os.path.abspath(os.path.join(EXPERIMENT_DIR, '../..'))
REPO_ROOT = os.path.join(WORKSPACE_ROOT, 'repos/TopoNet')

if WORKSPACE_ROOT not in sys.path:
    sys.path.insert(0, WORKSPACE_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from experiments.EXPERIMENT_1.utils.dataset import TopoNetDataset
from experiments.EXPERIMENT_1.utils.metrics import evaluate_batch
from experiments.EXPERIMENT_1.models.toponet_ablation import TopoNetAblationModel

# TopoNet Loss Suite
from cldice.cldice import soft_dice_cldice

# Check Betti Matching availability gracefully
HAS_BETTI = False
try:
    from utils.betti_loss import FastBettiMatchingLoss, FiltrationType
    HAS_BETTI = True
except Exception as e:
    # Notice for local Mac test or if C++ build is pending
    HAS_BETTI = False


def dice_loss_fn(pred, target, smooth=1e-5):
    """Standard Soft Multi-Class Dice Loss"""
    pred = torch.softmax(pred, dim=1)
    target_one_hot = target.float()
    intersection = (pred * target_one_hot).sum(dim=(2, 3))
    total = pred.sum(dim=(2, 3)) + target_one_hot.sum(dim=(2, 3))
    dice = (2.0 * intersection + smooth) / (total + smooth)
    return 1.0 - dice[:, 1:].mean()  # Exclude background class 0


def render_patient40_panels(img_t, gt_t, pred_t, filename, output_dir):
    """
    Renders 4-panel visual comparison: [RGB | GT Mask | Pred Mask | Error Map]
    """
    os.makedirs(output_dir, exist_ok=True)

    # 1. RGB
    rgb = (img_t.permute(1, 2, 0).cpu().numpy() * 255.0).clip(0, 255).astype(np.uint8)
    rgb_bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

    # 2. GT & Pred
    gt_class = torch.argmax(gt_t, dim=0).cpu().numpy().astype(np.uint8)
    pred_class = torch.argmax(pred_t, dim=0).cpu().numpy().astype(np.uint8)

    color_map = {
        0: (30, 30, 30),     # Background
        1: (0, 0, 255),      # Ridge: Red
        2: (0, 255, 0),      # Silhouette: Green
        3: (255, 0, 0),      # Falciform: Blue
    }

    gt_vis = np.zeros_like(rgb_bgr)
    pred_vis = np.zeros_like(rgb_bgr)

    for c, col in color_map.items():
        gt_vis[gt_class == c] = col
        pred_vis[pred_class == c] = col

    # 3. Error Map: Green = TP, Blue = FP, Red = FN
    error_vis = np.zeros_like(rgb_bgr)
    fg_gt = (gt_class > 0)
    fg_pred = (pred_class > 0)

    tp = np.logical_and(fg_gt, fg_pred)
    fp = np.logical_and(fg_pred, ~fg_gt)
    fn = np.logical_and(fg_gt, ~fg_pred)

    error_vis[tp] = (0, 255, 0)   # TP: Green
    error_vis[fp] = (255, 0, 0)   # FP: Blue
    error_vis[fn] = (0, 0, 255)   # FN: Red

    # Stitch into 1x4 panel
    h, w, _ = rgb_bgr.shape
    panel = np.zeros((h, w * 4, 3), dtype=np.uint8)
    panel[:, 0:w] = rgb_bgr
    panel[:, w:2*w] = gt_vis
    panel[:, 2*w:3*w] = pred_vis
    panel[:, 3*w:4*w] = error_vis

    # Add text banners
    cv2.putText(panel, "RGB Input", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
    cv2.putText(panel, "Ground Truth", (w + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
    cv2.putText(panel, "TopoNet Prediction", (2 * w + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
    cv2.putText(panel, "Error (G:TP, B:FP, R:FN)", (3 * w + 20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)

    save_name = os.path.splitext(filename)[0] + "_diag.png"
    cv2.imwrite(os.path.join(output_dir, save_name), panel)


def run_evaluation(model, dataloader, device, split_name='Val', save_patient40_dir=None):
    """Evaluates model, measures CUDA latency, and collects per-frame metrics."""
    model.eval()
    all_metrics = []
    latencies = []

    # Warmup for latency timing
    warmup_count = 0

    with torch.no_grad():
        for batch_idx, (images, depths, masks, filenames) in enumerate(tqdm(dataloader, desc=f"Evaluating {split_name}")):
            images = images.to(device)
            masks = masks.to(device)

            # CUDA Synchronized latency timing
            if device.type == 'cuda':
                torch.cuda.synchronize()
            start_t = time.perf_counter()

            logits, _ = model(images)

            if device.type == 'cuda':
                torch.cuda.synchronize()
            end_t = time.perf_counter()

            if warmup_count >= 5:
                latencies.append((end_t - start_t) * 1000.0 / images.size(0))
            else:
                warmup_count += 1

            # Batch metrics
            batch_m = evaluate_batch(logits, masks)
            for i, m in enumerate(batch_m):
                m['filename'] = filenames[i]
                m['patient'] = filenames[i].split('_')[1] if 'Patient_' in filenames[i] else 'unknown'
                all_metrics.append(m)

                # Patient 40 diagnostic rendering
                if save_patient40_dir and ('Patient_40_' in filenames[i] or '_40_' in filenames[i]):
                    render_patient40_panels(images[i], masks[i], logits[i], filenames[i], save_patient40_dir)

    # Aggregate summaries
    df = pd.DataFrame(all_metrics)
    mean_dice = float(df['macro_dice'].mean())
    mean_iou = float(df['macro_iou'].mean())
    mean_assd = float(df['macro_assd'].mean())

    mean_fg_dice = float(df['fg_dice'].mean())
    mean_fg_iou = float(df['fg_iou'].mean())
    mean_fg_assd = float(df['fg_assd'].mean())

    ridge_dice = float(df['ridge_dice'].mean())
    sil_dice = float(df['sil_dice'].mean())
    falc_dice = float(df['falc_dice'].mean())

    # Patient 40 subset
    p40_df = df[df['patient'] == '40']
    p40_dice = float(p40_df['macro_dice'].mean()) if len(p40_df) > 0 else 0.0
    p40_fg_dice = float(p40_df['fg_dice'].mean()) if len(p40_df) > 0 else 0.0
    p40_assd = float(p40_df['macro_assd'].mean()) if len(p40_df) > 0 else 80.0

    mean_latency = float(np.mean(latencies)) if len(latencies) > 0 else 0.0
    fps = float(1000.0 / mean_latency) if mean_latency > 0 else 0.0

    summary = {
        'split': split_name,
        'total_frames': len(df),
        'macro_dice': mean_dice,
        'macro_iou': mean_iou,
        'macro_assd': mean_assd,
        'fg_dice': mean_fg_dice,
        'fg_iou': mean_fg_iou,
        'fg_assd': mean_fg_assd,
        'ridge_dice': ridge_dice,
        'sil_dice': sil_dice,
        'falc_dice': falc_dice,
        'patient_40_dice': p40_dice,
        'patient_40_fg_dice': p40_fg_dice,
        'patient_40_assd': p40_assd,
        'patient_40_count': len(p40_df),
        'mean_latency_ms': mean_latency,
        'fps': fps,
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    }

    return summary, df


def main():
    parser = argparse.ArgumentParser(description="TopoNet Replication & Systematic Ablation Runner")
    parser.add_argument('--train_dir', type=str, default='data/L3D/Train', help="Path to Train directory")
    parser.add_argument('--val_dir', type=str, default='data/L3D/Val', help="Path to Val directory")
    parser.add_argument('--test_dir', type=str, default='data/L3D/Test', help="Path to Test directory")
    parser.add_argument('--depth_ckpt', type=str, default='checkpoints/depth_anything_v2_vitb.pth', help="Path to depth checkpoint")
    parser.add_argument('--ablation', type=str, default='full', 
                        choices=['full', 'baseline', 'wo_lper', 'wo_lcl', 'wo_lper_lcl', 'wo_btf'],
                        help="Ablation mode to execute")
    parser.add_argument('--epochs', type=int, default=100, help="Training epochs (paper: 100)")
    parser.add_argument('--batch_size', type=int, default=2, help="Micro-batch size")
    parser.add_argument('--accumulation_steps', type=int, default=2, help="Gradient accumulation steps (eff batch = batch * accum)")
    parser.add_argument('--lr', type=float, default=8e-5, help="Learning rate (paper: 8e-5)")
    parser.add_argument('--weight_decay', type=float, default=3e-5, help="Weight decay (paper: 3e-5)")
    parser.add_argument('--save_dir', type=str, default='results/toponet_full', help="Output results directory")
    parser.add_argument('--eval_splits', type=str, default='both', choices=['val', 'both'], help="Splits to evaluate at end")
    parser.add_argument('--smoke_test', action='store_true', help="Run 2-batch sanity check and exit")
    args = parser.parse_args()

    os.makedirs(args.save_dir, exist_ok=True)
    patient40_dir = os.path.join(args.save_dir, 'patient_40_diagnostics')

    device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
    print("=" * 80)
    print(f"🚀 TOPONET ABLATION RUNNER — EXPERIMENT_1")
    print(f"   Ablation Mode:        {args.ablation}")
    print(f"   Epochs:               {args.epochs}")
    print(f"   Micro Batch Size:     {args.batch_size} (Accumulation: {args.accumulation_steps} -> Effective Batch: {args.batch_size * args.accumulation_steps})")
    print(f"   Learning Rate:        {args.lr}")
    print(f"   Device:               {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Local'})")
    print(f"   Train Directory:      {args.train_dir}")
    print(f"   Val Directory:        {args.val_dir}")
    print(f"   Save Directory:       {args.save_dir}")
    print("=" * 80)

    # 1. Build Datasets
    train_dataset = TopoNetDataset(args.train_dir, mode='train')
    val_dataset = TopoNetDataset(args.val_dir, mode='val')

    train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=2, pin_memory=(device.type == 'cuda'))
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=(device.type == 'cuda'))

    test_loader = None
    if args.test_dir and os.path.exists(args.test_dir):
        test_dataset = TopoNetDataset(args.test_dir, mode='test')
        test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2)

    # 2. Build Model
    model = TopoNetAblationModel(ablation_mode=args.ablation, depth_path=args.depth_ckpt).to(device)

    # 3. Setup Loss Functions
    cl_dice_loss = soft_dice_cldice(exclude_background=True)
    betti_loss = None
    if args.ablation in ['full', 'wo_lcl', 'wo_btf']:
        if HAS_BETTI:
            betti_loss = FastBettiMatchingLoss(
                filtration_type=FiltrationType.SUPERLEVEL,
                num_processes=4,
                convert_to_one_vs_rest=False,
                ignore_background=True,
                push_unmatched_to_1_0=True,
                barcode_length_threshold=0.1,
                topology_weights=[0.5, 0.5]
            )
            print("✅ Betti Matching Loss initialized.")
        else:
            print("⚠️  BettiMatching C++ module not compiled. Running without Betti loss for this test.")

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=1e-6)

    best_val_dice = -1.0
    best_checkpoint_path = os.path.join(args.save_dir, "best_model.pth")

    # Smoke Test Short-Circuit
    if args.smoke_test:
        print("\n🧪 Running Local Smoke Test (1 iteration)...")
        model.train()
        for images, depths, masks, names in train_loader:
            images, masks = images.to(device), masks.to(device)
            optimizer.zero_grad()
            logits, _ = model(images)
            loss = dice_loss_fn(logits, masks)
            loss.backward()
            optimizer.step()
            print(f"   [Smoke Test] Forward/Backward Loss: {loss.item():.4f}")
            break

        print("\n🧪 Running Validation Smoke Test & Patient 40 Diagnostics...")
        val_summary, df_val = run_evaluation(model, val_loader, device, split_name='Val_Smoke', save_patient40_dir=patient40_dir)
        print(f"   [Smoke Test] Val Frames Evaluated: {len(df_val)} | Macro DSC: {val_summary['macro_dice']:.4f}")
        print("✅ Local Smoke Test Passed with Zero Errors!\n")
        return

    # 4. Main Training Loop
    total_iters = len(train_loader)
    for epoch in range(args.epochs):
        model.train()
        epoch_loss = 0.0
        optimizer.zero_grad()

        pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch [{epoch+1}/{args.epochs}]")
        for batch_idx, (images, depths, masks, names) in pbar:
            images = images.to(device)
            masks = masks.to(device)

            logits, _ = model(images)

            # Compute Loss based on Ablation Mode & Epoch
            if epoch >= 5 and args.ablation in ['full', 'wo_btf']:
                # Full TopoNet Dynamic Betti Warmup
                p = float(batch_idx + (epoch + 1) * total_iters) / (args.epochs * total_iters)
                alpha = (2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0) * 0.05
                seg_loss = cl_dice_loss(masks, logits)
                if betti_loss is not None:
                    b_out = betti_loss(logits, masks)
                    betti = b_out[0] if isinstance(b_out, (tuple, list)) else b_out
                else:
                    betti = 0.0
                raw_loss = betti * alpha + seg_loss * (1.0 - alpha)
            elif epoch >= 5 and args.ablation == 'wo_lper':
                raw_loss = cl_dice_loss(masks, logits)
            elif epoch >= 5 and args.ablation == 'wo_lcl':
                p = float(batch_idx + (epoch + 1) * total_iters) / (args.epochs * total_iters)
                alpha = (2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0) * 0.05
                d_loss = dice_loss_fn(logits, masks)
                if betti_loss is not None:
                    b_out = betti_loss(logits, masks)
                    betti = b_out[0] if isinstance(b_out, (tuple, list)) else b_out
                else:
                    betti = 0.0
                raw_loss = betti * alpha + d_loss * (1.0 - alpha)
            else:
                # Baseline, wo_lper_lcl, or Warmup epochs (0-4)
                raw_loss = dice_loss_fn(logits, masks)

            # Gradient Accumulation: scale loss
            loss = raw_loss / args.accumulation_steps
            loss.backward()

            epoch_loss += raw_loss.item()

            if (batch_idx + 1) % args.accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
                optimizer.step()
                optimizer.zero_grad()

            pbar.set_postfix({'loss': f"{raw_loss.item():.4f}"})

        scheduler.step()

        # Validation at epoch end
        if (epoch + 1) % 5 == 0 or (epoch + 1) == args.epochs:
            val_summary, df_val = run_evaluation(model, val_loader, device, split_name='Val', save_patient40_dir=patient40_dir)
            print(f"\n📊 Epoch {epoch+1} Val DSC: {val_summary['macro_dice']*100:.2f}% | IoU: {val_summary['macro_iou']*100:.2f}% | ASSD: {val_summary['macro_assd']:.2f}px | Patient 40 DSC: {val_summary['patient_40_dice']*100:.2f}%\n")

            if val_summary['macro_dice'] > best_val_dice:
                best_val_dice = val_summary['macro_dice']
                torch.save(model.state_dict(), best_checkpoint_path)
                print(f"🌟 Best model saved to {best_checkpoint_path} (DSC: {best_val_dice*100:.2f}%)")

    # 5. Final Comprehensive Evaluation
    print("\n" + "=" * 80)
    print("🏁 FINAL COMPREHENSIVE BENCHMARK EVALUATION")
    print("=" * 80)

    if os.path.exists(best_checkpoint_path):
        model.load_state_dict(torch.load(best_checkpoint_path, map_location=device))
        print(f"Loaded best checkpoint: {best_checkpoint_path}")

    # Evaluate Val
    final_val_summary, final_val_df = run_evaluation(model, val_loader, device, split_name='Val', save_patient40_dir=patient40_dir)
    final_val_df.to_csv(os.path.join(args.save_dir, "validation_per_frame_results.csv"), index=False)

    # Evaluate Test if requested
    final_test_summary = None
    if (args.eval_splits == 'both' or args.ablation == 'full') and test_loader is not None:
        final_test_summary, final_test_df = run_evaluation(model, test_loader, device, split_name='Test')
        final_test_df.to_csv(os.path.join(args.save_dir, "test_per_frame_results.csv"), index=False)

    # Save summary JSON
    results_json = {
        'ablation_mode': args.ablation,
        'epochs': args.epochs,
        'effective_batch_size': args.batch_size * args.accumulation_steps,
        'val_metrics': final_val_summary,
        'test_metrics': final_test_summary,
    }
    with open(os.path.join(args.save_dir, "summary_metrics.json"), 'w') as f:
        json.dump(results_json, f, indent=2)

    # Print Formatted Markdown Table for immediate viewing
    print("\n" + "=" * 80)
    print(f"🏆 BENCHMARK RESULTS SUMMARY: {args.ablation.upper()}")
    print("=" * 80)
    print(f"| Metric             | Validation (122 frames) | Test (109 frames) | Patient 40 Subset (Val) |")
    print(f"|:-------------------|:------------------------|:------------------|:------------------------|")
    print(f"| **Macro Mean DSC** | **{final_val_summary['macro_dice']*100:.2f}%**          | **{final_test_summary['macro_dice']*100 if final_test_summary else 0.0:.2f}%**       | **{final_val_summary['patient_40_dice']*100:.2f}%**              |")
    print(f"| **Mean IoU**       | {final_val_summary['macro_iou']*100:.2f}%          | {final_test_summary['macro_iou']*100 if final_test_summary else 0.0:.2f}%       | {final_val_summary.get('patient_40_iou', 0.0)*100:.2f}%              |")
    print(f"| **ASSD (px)**      | {final_val_summary['macro_assd']:.2f} px          | {final_test_summary['macro_assd'] if final_test_summary else 0.0:.2f} px       | {final_val_summary['patient_40_assd']:.2f} px              |")
    print(f"| **Ridge DSC**      | {final_val_summary['ridge_dice']*100:.2f}%          | {final_test_summary['ridge_dice']*100 if final_test_summary else 0.0:.2f}%       | --                      |")
    print(f"| **Silhouette DSC** | {final_val_summary['sil_dice']*100:.2f}%          | {final_test_summary['sil_dice']*100 if final_test_summary else 0.0:.2f}%       | --                      |")
    print(f"| **Falciform DSC**  | {final_val_summary['falc_dice']*100:.2f}%          | {final_test_summary['falc_dice']*100 if final_test_summary else 0.0:.2f}%       | --                      |")
    print(f"| **Latency / FPS**  | {final_val_summary['mean_latency_ms']:.1f} ms ({final_val_summary['fps']:.1f} FPS) | --                | Device: {final_val_summary['gpu_name']} |")
    print("=" * 80 + "\n")


if __name__ == '__main__':
    main()
''')

print("✅ All EXPERIMENT_1 modules written successfully.")


## Step 7: Run TopoNet Ablation Study
Configure and execute the desired ablation runs.

### Configuration Guide:
- `RUN_ALL_ABLATIONS = False`: Runs **Run 1.0 (Full TopoNet)** on both Validation and Test splits first.
- `RUN_ALL_ABLATIONS = True`: Runs all 6 paper ablation configurations sequentially:
  1. `full`: Full TopoNet (ResNet34 + Depth + DSCNet + BTF + clDice + Betti)
  2. `baseline`: Baseline (ResNet34 + Depth + Conv + Simple Concat + Soft Dice)
  3. `wo_lper`: w/o $L_{per}$ (clDice only)
  4. `wo_lcl`: w/o $L_{cl}$ (Betti only)
  5. `wo_lper_lcl`: w/o $L_{per}$ & $L_{cl}$ (Soft Dice only)
  6. `wo_btf`: w/o BTF (Simple Concat + clDice + Betti)


In [ ]:
# ==============================================================================
# 🎛️ EXPERIMENT RUNNER CONFIGURATION
# ==============================================================================
RUN_ALL_ABLATIONS = False    # Set to True to run all 6 paper ablations in sequence
RUN_FULL_TEST_SPLIT = True   # Evaluate Full TopoNet on Test split (Table 1 SOTA)
EPOCHS = 100                 # Standard paper epochs (100)
BATCH_SIZE = 2               # Micro-batch 2 + Accumulation 2 = Effective Batch 4 (fits 16GB T4)
ACCUMULATION_STEPS = 2
LR = 8e-5
WEIGHT_DECAY = 3e-5

if RUN_ALL_ABLATIONS:
    ABLATION_LIST = ['full', 'baseline', 'wo_lper', 'wo_lcl', 'wo_lper_lcl', 'wo_btf']
else:
    # Test Run 1.0 (Full TopoNet) first
    ABLATION_LIST = ['full']

print(f"📋 Ablation queue: {ABLATION_LIST}")
print(f"📊 Training parameters: {EPOCHS} epochs, lr={LR}, effective batch size={BATCH_SIZE * ACCUMULATION_STEPS}")

# Execute runs sequentially
for abl in ABLATION_LIST:
    run_save_dir = f"/kaggle/working/results/run_{abl}"
    os.makedirs(run_save_dir, exist_ok=True)
    
    print("\n" + "=" * 80)
    print(f"🚀 LAUNCHING RUN: {abl.upper()}")
    print(f"📁 Output Directory: {run_save_dir}")
    print("=" * 80)
    
    cmd = [
        "python", "/kaggle/working/experiments/EXPERIMENT_1/scripts/train_toponet.py",
        "--train_dir", train_dir,
        "--val_dir", val_dir,
        "--test_dir", test_dir,
        "--depth_weights", depth_ckpt,
        "--save_dir", run_save_dir,
        "--ablation", abl,
        "--epochs", str(EPOCHS),
        "--batch_size", str(BATCH_SIZE),
        "--accumulation_steps", str(ACCUMULATION_STEPS),
        "--lr", str(LR),
        "--weight_decay", str(WEIGHT_DECAY),
        "--eval_splits", "both" if (abl == 'full' and RUN_FULL_TEST_SPLIT) else "val"
    ]
    
    subprocess.run(cmd, check=True)


## Step 8: Summary Benchmark Table & Results Download
Collect all metric summaries, verify Patient 40 diagnostic plots, and package into `EXPERIMENT_1_RESULTS.zip`.


In [ ]:
import json
import glob
import pandas as pd
from IPython.display import display, Markdown, FileLink

print("=" * 80)
print("📊 EMPIRICAL ABLATION RESULTS (LIVE FROM EXPERIMENT RUNS)")
print("=" * 80)

summary_files = sorted(glob.glob('/kaggle/working/results/run_*/summary_metrics.json'))

if not summary_files:
    print("⚠️ No summary_metrics.json files found. Have you executed the training cell above?")
else:
    rows = []
    for sf in summary_files:
        try:
            with open(sf, 'r') as f:
                data = json.load(f)
            abl = data.get('ablation_mode', 'unknown').upper()
            val_m = data.get('val_metrics', {})
            test_m = data.get('test_metrics', {})
            
            row = {
                'Ablation Mode': abl,
                'Val Macro DSC': f"{val_m.get('macro_dice', 0)*100:.2f}%",
                'Val FG DSC': f"{val_m.get('fg_dice', 0)*100:.2f}%",
                'Val IoU': f"{val_m.get('macro_iou', 0)*100:.2f}%",
                'Val ASSD (px)': f"{val_m.get('macro_assd', 0):.2f}",
                'Ridge DSC': f"{val_m.get('ridge_dice', 0)*100:.2f}%",
                'Silhouette DSC': f"{val_m.get('sil_dice', 0)*100:.2f}%",
                'Falciform DSC': f"{val_m.get('falc_dice', 0)*100:.2f}%",
                'Patient 40 DSC': f"{val_m.get('patient_40_dice', 0)*100:.2f}%",
                'Test Macro DSC': f"{test_m.get('macro_dice', 0)*100:.2f}%" if test_m else "--",
                'Latency (ms)': f"{val_m.get('mean_latency_ms', 0):.1f}"
            }
            rows.append(row)
        except Exception as e:
            print(f"Error reading {sf}: {e}")

    df_results = pd.DataFrame(rows)
    display(Markdown(df_results.to_markdown(index=False)))

# Check Patient 40 visual diagnostics
p40_images = glob.glob('/kaggle/working/results/**/visualizations_patient40/*.png', recursive=True)
print(f"\n🔍 Patient 40 diagnostic panels generated: {len(p40_images)}")

# Package all results into ZIP
zip_dest = '/kaggle/working/EXPERIMENT_1_RESULTS.zip'
print(f"📦 Packaging results into {zip_dest}...")
!zip -q -r {zip_dest} /kaggle/working/results/

if os.path.exists(zip_dest):
    print(f"✅ ZIP Archive ready: {zip_dest} ({os.path.getsize(zip_dest) / (1024**2):.2f} MB)")
    print("👉 Download the results directly from Kaggle output pane or click below:")
    display(FileLink('EXPERIMENT_1_RESULTS.zip'))
